# Imprinting Analysis with Control Reference: GSE74738

This notebook runs imprintomeR on **GSE74738**, an Illumina 450K cohort of placentas and related samples with a large group of normal controls. Because the cohort has its own controls, scores can be **centred on the controls** instead of the fixed 0.5 reference, and the per-ICR calls use both of their criteria (|deviation| ≥ 0.075 and |z| ≥ 3).

The cohort contains genetically defined parent-of-origin imbalances, which give expected directions:

| Group | Samples | Parental genomes | Expected shift |
|---|---:|---|---|
| control | 53 | balanced | none (reference) |
| CHM (complete hydatidiform mole) | 1 | paternal only (androgenetic) | paternal gain + maternal loss |
| diandric triploid | 5 | 2 paternal : 1 maternal | towards paternal gain |
| digynic triploid | 5 | 1 paternal : 2 maternal | towards maternal gain |
| IUGR, LOPET, chromosomally normal miscarriage, Ab. MSS | 15 | balanced | none expected |

Placental imprinting includes placenta-specific DMRs, so results describe placenta and should not be read as blood or adult-tissue behaviour.

You will:

1. download the raw IDATs and metadata from GEO (or reuse a QC set from the command-line workflow);
2. run methylation-array QC;
3. score imprinting against the fixed 0.5 reference and against the cohort's controls;
4. score each ICR and summarize altered ICRs per sample;
5. add bootstrap intervals, the origin permutation test and a control-calibrated IDS cutoff;
6. score the chr11p15 and chr20q13 locus panels;
7. visualize and export the results.

## Phase 1: Setup

In [ ]:
library(imprintomeR)
library(dplyr)

set.seed(2026)
options(repr.plot.width = 10, repr.plot.height = 8)
packageVersion("imprintomeR")

## Phase 2: Data

Choose **one** of the two options.

### Option A: raw IDATs from GEO

The raw archive (`GSE74738_RAW.tar`) is downloaded once into `GSE74738_raw/`. This cell is the slow one; skip it when you use Option B.

In [ ]:
library(GEOquery)
geo_gse <- "GSE74738"
raw_dir <- "GSE74738_raw"
dir.create(raw_dir, showWarnings = FALSE)

gse   <- getGEO(geo_gse, GSEMatrix = TRUE, getGPL = FALSE)
pdata <- pData(gse[[1]])

supp <- getGEOSuppFiles(geo_gse, baseDir = raw_dir)
untar(rownames(supp)[grepl("RAW\\.tar$", rownames(supp))], exdir = file.path(raw_dir, "idats"))

Build the QC metadata: one row per sample with `Sample_Name`, `Sample_Group` (from the GEO `status/group` field) and `Basename`, the absolute IDAT path prefix.

In [ ]:
idat_grn <- list.files(file.path(raw_dir, "idats"), pattern = "_Grn\\.idat(\\.gz)?$", full.names = TRUE)
idat_map <- data.frame(
  Basename      = normalizePath(sub("_Grn\\.idat(\\.gz)?$", "", idat_grn)),
  geo_accession = sub("_.*$", "", basename(idat_grn)),
  stringsAsFactors = FALSE
)

group_col <- grep("group:ch1$", colnames(pdata), value = TRUE)[1]
meta_qc <- data.frame(
  Sample_Name   = pdata$geo_accession,
  Sample_Group  = trimws(pdata[[group_col]]),
  Title         = pdata$title,
  stringsAsFactors = FALSE
)
meta_qc$Basename <- idat_map$Basename[match(meta_qc$Sample_Name, idat_map$geo_accession)]
meta_qc <- meta_qc[!is.na(meta_qc$Basename), ]

table(meta_qc$Sample_Group)

Run QC (450K). `Final.QC` is PASS when at least 95% of probes are detected at p < 0.05 and the bisulfite conversion I control metrics are at least 1.

In [ ]:
qcset <- runMethQC(meta_qc, platform = "450K", outdir = "GSE74738_qc", prefix = "450k")

### Option B: QC set from the command-line workflow

If you already ran `run_meth_QC.R` (for example `--platform 450K --outdir qc_results`), load its QC set instead.

In [ ]:
qcset <- readRDS("qc_results/450k/data/450k_qcset.rds")

## Phase 3: QC review

In [ ]:
qc_matrix <- qc_tables(qcset)[["QC_matrix"]]
table(qc_matrix$Final.QC)
qc_params(qcset)$normalization

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 5)
plot(qcset, type = "qc_bar")
plot(qcset, type = "probe_coverage")

## Phase 4: ImprintomeSet and sample-level scores

In [ ]:
qc_pass <- subsetMethQC(qcset, final_qc = "PASS")
x <- as.ImprintomeSet(qc_pass, genome = "hg38")
x
table(meta(x)$Sample_Group)

Score the `selected` panel (gametic ICRs by default) twice: against the fixed 0.5 reference, and centred on the cohort's controls (`reference_group = "control"`; each control is scored leave-one-out). `icr = TRUE` adds the per-ICR tables to the control-centred run.

In [ ]:
x <- runImprintome(x, probeset = "selected")                                         # fixed 0.5 reference
x <- runImprintome(x, probeset = "selected", reference_group = "control", icr = TRUE)  # centred on controls
names(results(x))

In [ ]:
group_summary <- function(tab, reference) {
  tab %>%
    group_by(Sample_Group) %>%
    summarise(n = n(), median_IDS = round(median(IDS), 3), max_IDS = round(max(IDS), 3),
              top_mechanism = names(sort(table(Mechanism), decreasing = TRUE))[1], .groups = "drop") %>%
    mutate(reference = reference)
}
res <- results(x)
bind_rows(group_summary(res[["AnalyzeImprintStatus.selected"]], "fixed 0.5"),
          group_summary(res[["AnalyzeImprintStatus.selected.ref-controls"]], "controls"))

Samples with a parent-of-origin imbalance: the androgenetic mole falls in Pat-Gain+Mat-Loss; triploids shift less, towards the paternal (diandric) or maternal (digynic) side. Compare their angles.

In [ ]:
status <- res[["AnalyzeImprintStatus.selected.ref-controls"]]
status %>%
  filter(Sample_Group %in% c("CHM", "diandric triploid", "digynic triploid")) %>%
  select(Sample_Name, Sample_Group, IDS, Angle, Mechanism) %>%
  arrange(Sample_Group, Angle)

## Phase 5: Per-ICR scores

Each ICR is scored on its own: the median of (beta − control median) over its probes, called altered when |deviation| ≥ 0.075 and |z| ≥ 3 against the controls. The summary counts altered ICRs per sample and flags multi-locus disturbance (MLID) when they lie on at least two chromosomes. Per-ICR scores cover all annotated ICRs, secondary DMRs included.

In [ ]:
icr_table   <- res[["ICRStatus.selected"]]
icr_summary <- res[["ICRSummary.selected"]]

icr_summary %>%
  group_by(Sample_Group) %>%
  summarise(n = n(), median_altered = median(n_altered), max_altered = max(n_altered),
            median_gain = median(n_gain), median_loss = median(n_loss), MLID = sum(MLID), .groups = "drop")

In [ ]:
# Median per-ICR deviation by annotated parental origin in the triploids
icr_table %>%
  filter(Sample_Group %in% c("diandric triploid", "digynic triploid")) %>%
  group_by(Sample_Group, ORIGIN) %>%
  summarise(median_deviation = round(median(deviation), 3), .groups = "drop")

# Count gametic ICRs only
SummarizeICRStatus(icr_table, dmr_type = "gametic") %>%
  select(Sample_Name, Sample_Group, n_icrs, n_altered, MLID) %>%
  arrange(desc(n_altered)) %>%
  head(15)

## Phase 6: Uncertainty and calibration

With a control reference, all three options apply:

- `n_boot`: bootstrap intervals for IDS and Angle, and `Mechanism_support`, the fraction of replicates in the reported sector (use ≥ 1,000 replicates for reporting; 200 keep this fast);
- `n_perm`: a permutation test of whether the deviation depends on the annotated parental origin (`origin_perm_p`);
- `CalibrateIDS()`: an IDS cutoff from the controls' own IDS distribution, here its 99th percentile.

In [ ]:
x <- runImprintome(x, probeset = "selected", reference_group = "control",
                   n_boot = 200, n_perm = 200, seed = 2026, result_name = "selected.uncertainty")
unc <- results(x)[["selected.uncertainty"]]
unc %>%
  filter(Sample_Group != "control") %>%
  select(Sample_Name, Sample_Group, IDS, IDS_lower, IDS_upper, Angle, Mechanism, Mechanism_support, origin_perm_p) %>%
  arrange(desc(IDS)) %>%
  head(15)

In [ ]:
cal <- CalibrateIDS(unc, controls = "control", method = "quantile", probs = 0.99)
attr(cal, "calibration")
table(cal$Sample_Group, cal$Status_calibrated)

## Phase 7: Locus panels (chr11p15 and chr20q13)

In [ ]:
x <- runImprintome(x, probeset = "chr11p15", reference_group = "control")
x <- runImprintome(x, probeset = "chr20q13", reference_group = "control")
res <- results(x)
bind_rows(group_summary(res[["AnalyzeImprintStatus.chr11p15.ref-controls"]], "chr11p15"),
          group_summary(res[["AnalyzeImprintStatus.chr20q13.ref-controls"]], "chr20q13"))

## Phase 8: Visualizations

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 10)
plot(x, plot_type = "polar", probeset = "selected",
     result_name = "AnalyzeImprintStatus.selected.ref-controls",
     title = "GSE74738: selected panel, centred on controls")

In [ ]:
options(repr.plot.width = 14, repr.plot.height = 10)
plot(x, plot_type = "icr_heatmap", probeset = "selected")

In [ ]:
chm <- status$Sample_Name[status$Sample_Group == "CHM"][1]
options(repr.plot.width = 17, repr.plot.height = 17)
plot(x, plot_type = "radar", probeset = "selected", sample_id = chm, title = paste(chm, "(CHM)"))

In [ ]:
options(repr.plot.width = 14, repr.plot.height = 6)
plot(x, plot_type = "beeswarm_origin", probeset = "selected", SAMPLEID = "Sample_Name")

## Phase 9: Export

In [ ]:
manifest <- export(x, outdir = "GSE74738_imprintome_output", prefix = "GSE74738",
                   save_plots = FALSE, overwrite = TRUE)
head(manifest)

## Session information

In [ ]:
local({ si <- sessionInfo(); si$BLAS <- basename(si$BLAS); si$LAPACK <- basename(si$LAPACK); si })